# Import Libraries

In [14]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [8]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from datasets import Dataset, DatasetDict
import matplotlib.pyplot as plt
from wordcloud import WordCloud
import re
import torch
from torch import nn
from transformers import Trainer, TrainingArguments, AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import f1_score
import ast

# PREPARATION

In [5]:
df_sampled = pd.read_csv('df_sampled_cleaned.csv')

In [6]:
df_sampled.head()

,id,title,abstract,categories,categories_list,labels
0,1711.00614,A Multimodal Anomaly Detector for Robot-Assist...,The detection of anomalous executions is valua...,cs.RO cs.LG,"['cs.RO', 'cs.LG']","[1.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, ..."
1,2401.00701,Towards Efficient and Effective Text-to-Video ...,"In recent years, text-to-video retrieval metho...",cs.CV,['cs.CV'],"[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
2,2205.00348,End-to-End Signal Classification in Signed Cum...,This paper presents a new end-to-end signal cl...,eess.SP cs.LG,"['cs.LG', 'eess.SP']","[1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
3,2210.00069,Topological Singularity Detection at Multiple ...,"The manifold hypothesis, which assumes that da...",cs.LG cs.AI math.AT stat.ML,"['stat.ML', 'cs.AI', 'cs.LG', 'math.AT']","[1.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."
4,2412.17068,A Plug-and-Play Natural Language Rewriter for ...,Existing Natural Language to SQL (NL2SQL) solu...,cs.DB,['cs.DB'],"[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, ..."


In [9]:
# ==========================================
# 3. CALCULATE EXACT CLASS WEIGHTS
# ==========================================
# We calculate weights AFTER dropping junk rows so the math is perfect
# --- THE FIX ---
# Convert the literal strings back into actual Python lists
df_sampled['labels'] = df_sampled['labels'].apply(ast.literal_eval)
labels_matrix = np.array(df_sampled['labels'].tolist())
num_samples = labels_matrix.shape[0]

positive_counts = labels_matrix.sum(axis=0)
negative_counts = num_samples - positive_counts
pos_weights = negative_counts / (positive_counts + 1e-5) # 1e-5 prevents division by zero

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
pos_weights_tensor = torch.tensor(pos_weights, dtype=torch.float32).to(device)
print("Class weights calculated successfully!")

Class weights calculated successfully!


In [10]:
# ==========================================
# 4. CREATE THE CUSTOM WEIGHTED TRAINER
# ==========================================
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        
        # Apply our exact mathematical weights to the loss function
        loss_fct = nn.BCEWithLogitsLoss(pos_weight=pos_weights_tensor)
        loss = loss_fct(logits, labels)
        
        return (loss, outputs) if return_outputs else loss

# ==========================================
# 5. SPLIT, TOKENIZE, AND FORMAT
# ==========================================
print("Splitting and tokenizing...")

# --- THE FIX: Drop all unnecessary metadata columns like 'id' and 'title' ---
# Keep only what the model needs: the text and the labels.
df_clean = df_sampled[['abstract', 'labels']].copy()

train_df, temp_df = train_test_split(df_clean, test_size=0.2, random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42)

hg_dataset = DatasetDict({
    'train': Dataset.from_pandas(train_df, preserve_index=False),
    'validation': Dataset.from_pandas(val_df, preserve_index=False),
    'test': Dataset.from_pandas(test_df, preserve_index=False)
})

tokenizer = AutoTokenizer.from_pretrained("allenai/scibert_scivocab_uncased")

def tokenize_function(examples):
    # Using the 300 token limit we discovered during EDA!
    return tokenizer(examples['abstract'], padding="max_length", truncation=True, max_length=300)

tokenized_datasets = hg_dataset.map(tokenize_function, batched=True)

# Keep only the columns the PyTorch model actually needs
cols_to_keep = ['input_ids', 'attention_mask', 'labels', 'token_type_ids']
cols_to_remove = [col for col in tokenized_datasets['train'].column_names if col not in cols_to_keep]
tokenized_datasets = tokenized_datasets.remove_columns(cols_to_remove)
tokenized_datasets.set_format('torch')

print("Tokenization successful! No ArrowTypeErrors.")



Splitting and tokenizing...


c:\Users\User\miniconda3\envs\final\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


Map:   0%|          | 0/142267 [00:00<?, ? examples/s]

Map:   0%|          | 0/17783 [00:00<?, ? examples/s]

Map:   0%|          | 0/17784 [00:00<?, ? examples/s]

Tokenization successful! No ArrowTypeErrors.


# Load and Training

In [11]:

# ==========================================
# 6. INITIALIZE MODEL
# ==========================================
print("Loading SciBERT Model...")
model = AutoModelForSequenceClassification.from_pretrained(
    "allenai/scibert_scivocab_uncased", 
    num_labels=40, 
    problem_type="multi_label_classification"
).to(device)



Loading SciBERT Model...


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at allenai/scibert_scivocab_uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [17]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.sigmoid(torch.Tensor(logits)).numpy()
    predictions = (probs > 0.5).astype(int)
    macro_f1 = f1_score(labels, predictions, average='macro', zero_division=0)
    return {'macro_f1': macro_f1}

training_args = TrainingArguments(
    output_dir='./scibert-weighted-cs',
    eval_strategy="epoch",            
    save_strategy="epoch",                  
    learning_rate=2e-5,                     
    # --- RTX 3050 OPTIMIZATIONS ---
    per_device_train_batch_size=4,        # Shrunk from 16 to 4 to fit in 4GB VRAM
    gradient_accumulation_steps=4,        # 4 (batch size) * 4 (steps) = 16 effective batch size!
    per_device_eval_batch_size=8,         # Eval uses less memory since it doesn't calculate gradients
    fp16=True,                            # CRITICAL for RTX 30 series: halves VRAM usage and doubles speed
    # ------------------------------
    num_train_epochs=3,                     
    weight_decay=0.01,                      
    load_best_model_at_end=True,            
    metric_for_best_model="macro_f1",       
)

# Initialize our custom class!
trainer = WeightedTrainer(
    model=model,                            
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    compute_metrics=compute_metrics
)

print("\n🚀 Pipeline fully assembled! Run `trainer.train()` to begin.")

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [13]:
trainer.train()

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.
wandb: Currently logged in as: galnoel (galnoel-universitas-sam-ratulangi) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


  0%|          | 0/26673 [00:00<?, ?it/s]

RuntimeError: CUDA error: out of memory
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.


In [ ]:
# Save the model weights, configuration, and our custom weighted trainer state
trainer.save_model("./my_final_scibert_model")

# Save the tokenizer so it processes future text exactly the same way
tokenizer.save_pretrained("./my_final_scibert_model")

print("Model and tokenizer safely saved to disk!")

In [ ]:
import torch

# 1. Put the model in evaluation mode (turns off training-specific behaviors like dropout)
model.eval()

# 2. Write a brand new abstract to test it on
test_abstract = "In this paper, we propose a novel convolutional neural network architecture equipped with attention mechanisms for real-time object detection and instance segmentation."

# 3. Tokenize the input just like we did for training
inputs = tokenizer(
    test_abstract, 
    return_tensors="pt", 
    padding="max_length", 
    truncation=True, 
    max_length=300
).to(device) # Move the tokens to the GPU

# 4. Pass the text through the model
with torch.no_grad(): # Tells PyTorch not to calculate gradients (saves memory)
    outputs = model(**inputs)

# 5. Convert raw logits to probabilities, then to 1s and 0s
probs = torch.sigmoid(outputs.logits).squeeze().cpu().numpy()
binary_predictions = (probs > 0.5).astype(int)

# 6. Translate the 1s and 0s back into human-readable category names!
# (This uses the mlb_40_cs variable we created earlier)
predicted_categories = mlb_40_cs.classes_[binary_predictions == 1]

print(f"Abstract: {test_abstract}\n")
print(f"Predicted Categories: {predicted_categories}")